# Public review copy — MIS581 Capstone

This notebook is a sanitized publication copy of an executed development notebook. Local Windows paths were replaced with relative placeholders and embedded Census API configuration was changed to use the `CENSUS_API_KEY` environment variable. Historical error outputs were omitted; successful saved outputs were retained. The notebook has **not** been independently re-executed as a clean installation. It may require original source data, a compatible Python environment, and (for Notebook 08) PostgreSQL. Use the project README to understand dependencies and expected folder layout.


In [2]:
import os
import pandas as pd
import requests

# Select a historical year
year = 2019

# Census API key

api_key = os.getenv("CENSUS_API_KEY", "")
# Census ACS 1-year endpoint
base_url = f"https://api.census.gov/data/{year}/acs/acs1"

# Request population and median household income
params = {
    "get": "NAME,B01003_001E,B19013_001E",
    "for": "metropolitan statistical area/micropolitan statistical area:*",
    "key": api_key
}

response = requests.get(base_url, params=params, timeout=30)

print("Status Code:", response.status_code)
print("Content Type:", response.headers.get("Content-Type"))
print(response.text[:500])

Status Code: 200
Content Type: application/json;charset=utf-8
[["NAME","B01003_001E","B19013_001E","metropolitan statistical area/micropolitan statistical area"],
["Aberdeen, WA Micro Area","75061","61026","10140"],
["Abilene, TX Metro Area","171795","54808","10180"],
["Adrian, MI Micro Area","98451","53865","10300"],
["Aguadilla-Isabela, PR Metro Area","288877","16311","10380"],
["Akron, OH Metro Area","703479","57158","10420"],
["Alamogordo, NM Micro Area","67490","39371","10460"],
["Albany, GA Metro Area","147384","40625","10500"],
["Albany-Lebanon, OR 


In [3]:
# Convert Census JSON response into a DataFrame
data = response.json()

df_census_2019 = pd.DataFrame(
    data[1:],
    columns=data[0]
)

# Rename columns for easier analysis
df_census_2019 = df_census_2019.rename(columns={
    "NAME": "metro",
    "B01003_001E": "population",
    "B19013_001E": "median_household_income",
    "metropolitan statistical area/micropolitan statistical area": "cbsa"
})

# Convert numerical columns
df_census_2019["population"] = pd.to_numeric(
    df_census_2019["population"],
    errors="coerce"
)

df_census_2019["median_household_income"] = pd.to_numeric(
    df_census_2019["median_household_income"],
    errors="coerce"
)

# Add year for future integration
df_census_2019["year"] = year

# Review results
print("Dataset Shape:", df_census_2019.shape)

print("\nColumn Types:")
print(df_census_2019.dtypes)

print("\nMissing Values:")
print(df_census_2019.isna().sum())

print("\nFirst Five Records:")
display(df_census_2019.head())

Dataset Shape: (518, 5)

Column Types:
metro                      object
population                  int64
median_household_income     int64
cbsa                       object
year                        int64
dtype: object

Missing Values:
metro                      0
population                 0
median_household_income    0
cbsa                       0
year                       0
dtype: int64

First Five Records:


,metro,population,median_household_income,cbsa,year
0,"Aberdeen, WA Micro Area",75061,61026,10140,2019
1,"Abilene, TX Metro Area",171795,54808,10180,2019
2,"Adrian, MI Micro Area",98451,53865,10300,2019
3,"Aguadilla-Isabela, PR Metro Area",288877,16311,10380,2019
4,"Akron, OH Metro Area",703479,57158,10420,2019


In [4]:
# Step 3: Validate Census ACS 2019 data

# 1. Descriptive statistics
print("Population and Income Statistics:")
display(
    df_census_2019[
        ["population", "median_household_income"]
    ].describe()
)

# 2. Check for invalid or suppressed values
print("\nZero or Negative Values:")

print(
    "Population:",
    (df_census_2019["population"] <= 0).sum()
)

print(
    "Median Household Income:",
    (df_census_2019["median_household_income"] <= 0).sum()
)

# 3. Count metropolitan and micropolitan areas
print("\nGeographic Area Counts:")

metro_count = df_census_2019["metro"].str.contains(
    "Metro Area", case=False, na=False
).sum()

micro_count = df_census_2019["metro"].str.contains(
    "Micro Area", case=False, na=False
).sum()

print("Metropolitan Areas:", metro_count)
print("Micropolitan Areas:", micro_count)

print(
    "Total Classified:",
    metro_count + micro_count
)

# 4. Check duplicate CBSA-year combinations
print("\nDuplicate CBSA-Year Records:")

duplicates = df_census_2019.duplicated(
    subset=["cbsa", "year"]
).sum()

print("Duplicates:", duplicates)

Population and Income Statistics:


,population,median_household_income
count,5.180000e+02,518.000000
mean,5.737360e+05,58663.976834
std,1.436660e+06,13135.531565
min,5.591600e+04,15695.000000
25%,1.008312e+05,50468.500000
50%,1.692670e+05,57467.000000
75%,4.202892e+05,65001.750000
max,1.921618e+07,130865.000000



Zero or Negative Values:
Population: 0
Median Household Income: 0

Geographic Area Counts:
Metropolitan Areas: 392
Micropolitan Areas: 126
Total Classified: 518

Duplicate CBSA-Year Records:
Duplicates: 0


In [5]:
# Step 4: Filter to metropolitan statistical areas

df_census_2019_msa = df_census_2019[
    df_census_2019["metro"].str.endswith(
        "Metro Area", na=False
    )
].copy()

# Reset the index
df_census_2019_msa.reset_index(
    drop=True,
    inplace=True
)

# Review filtered dataset
print("Original Dataset:", df_census_2019.shape)

print("Metropolitan Dataset:", df_census_2019_msa.shape)

print(
    "\nUnique CBSA Codes:",
    df_census_2019_msa["cbsa"].nunique()
)

print(
    "\nDuplicate CBSA-Year Records:",
    df_census_2019_msa.duplicated(
        subset=["cbsa", "year"]
    ).sum()
)

print("\nFirst Five Metropolitan Records:")
display(df_census_2019_msa.head())

Original Dataset: (518, 5)
Metropolitan Dataset: (392, 5)

Unique CBSA Codes: 392

Duplicate CBSA-Year Records: 0

First Five Metropolitan Records:


,metro,population,median_household_income,cbsa,year
0,"Abilene, TX Metro Area",171795,54808,10180,2019
1,"Aguadilla-Isabela, PR Metro Area",288877,16311,10380,2019
2,"Akron, OH Metro Area",703479,57158,10420,2019
3,"Albany, GA Metro Area",147384,40625,10500,2019
4,"Albany-Lebanon, OR Metro Area",129749,61488,10540,2019


In [6]:
import time

# Step 5: Historical ACS 1-Year Availability Check

years = range(2005, 2026)

availability_results = []
census_historical_data = {}

for yr in years:

    print(f"Checking {yr}...", end=" ")

    url = f"https://api.census.gov/data/{yr}/acs/acs1"

    params = {
        "get": "NAME,B01003_001E,B19013_001E",
        "for": "metropolitan statistical area/micropolitan statistical area:*",
        "key": api_key
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        # Confirm the response is JSON
        if "application/json" not in response.headers.get(
            "Content-Type", ""
        ):
            raise ValueError("Response was not JSON")

        data = response.json()

        df = pd.DataFrame(
            data[1:],
            columns=data[0]
        )

        df = df.rename(columns={
            "NAME": "metro",
            "B01003_001E": "population",
            "B19013_001E": "median_household_income",
            "metropolitan statistical area/micropolitan statistical area": "cbsa"
        })

        # Filter metropolitan areas
        df = df[
            df["metro"].str.endswith(
                "Metro Area", na=False
            )
        ].copy()

        # Convert numeric fields
        for col in ["population", "median_household_income"]:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

        df["year"] = yr

        # Preserve retrieved data
        census_historical_data[yr] = df

        invalid_population = (
            df["population"].isna() |
            (df["population"] <= 0)
        ).sum()

        invalid_income = (
            df["median_household_income"].isna() |
            (df["median_household_income"] <= 0)
        ).sum()

        availability_results.append({
            "year": yr,
            "status": "Available",
            "msa_count": len(df),
            "invalid_population": invalid_population,
            "invalid_income": invalid_income,
            "duplicate_cbsa_year": df.duplicated(
                subset=["cbsa", "year"]
            ).sum()
        })

        print(f"Success — {len(df)} MSAs")

    except (requests.RequestException, ValueError, KeyError) as e:

        availability_results.append({
            "year": yr,
            "status": "Unavailable / Error",
            "msa_count": None,
            "invalid_population": None,
            "invalid_income": None,
            "duplicate_cbsa_year": None
        })

        print(f"Failed — {str(e)[:100]}")

    # Brief pause between requests
    time.sleep(0.5)


# Create summary DataFrame
df_availability = pd.DataFrame(availability_results)

print("\nHistorical ACS Availability Summary:")
display(df_availability)

print("\nYears Successfully Retrieved:")
print(
    df_availability.loc[
        df_availability["status"] == "Available",
        "year"
    ].tolist()
)

print("\nYears Requiring Investigation:")
print(
    df_availability.loc[
        df_availability["status"] != "Available",
        "year"
    ].tolist()
)

Checking 2005... Success — 366 MSAs
Checking 2006... Success — 367 MSAs
Checking 2007... Success — 369 MSAs
Checking 2008... Success — 369 MSAs
Checking 2009... Success — 374 MSAs
Checking 2010... Success — 374 MSAs
Checking 2011... Success — 374 MSAs
Checking 2012... Success — 374 MSAs
Checking 2013... Success — 388 MSAs
Checking 2014... Success — 388 MSAs
Checking 2015... Success — 388 MSAs
Checking 2016... Success — 389 MSAs
Checking 2017... Success — 389 MSAs
Checking 2018... Success — 390 MSAs
Checking 2019... Success — 392 MSAs
Checking 2020... Failed — 404 Client Error:  for url: https://api.census.gov/data/2020/acs/acs1?get=NAME%2CB01003_001E%2CB1901
Checking 2021... Success — 392 MSAs
Checking 2022... Success — 392 MSAs
Checking 2023... Success — 393 MSAs
Checking 2024... Success — 393 MSAs
Checking 2025... Failed — 404 Client Error:  for url: https://api.census.gov/data/2025/acs/acs1?get=NAME%2CB01003_001E%2CB1901

Historical ACS Availability Summary:


,year,status,msa_count,invalid_population,invalid_income,duplicate_cbsa_year
0,2005,Available,366.0,0.0,0.0,0.0
1,2006,Available,367.0,0.0,0.0,0.0
2,2007,Available,369.0,0.0,0.0,0.0
3,2008,Available,369.0,0.0,0.0,0.0
4,2009,Available,374.0,0.0,0.0,0.0
5,2010,Available,374.0,0.0,0.0,0.0
6,2011,Available,374.0,0.0,0.0,0.0
7,2012,Available,374.0,0.0,0.0,0.0
8,2013,Available,388.0,0.0,0.0,0.0
9,2014,Available,388.0,0.0,0.0,0.0



Years Successfully Retrieved:
[2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024]

Years Requiring Investigation:
[2020, 2025]


In [7]:
# Step 6: Validate historical coverage for our seven selected MSAs

selected_msas = {
    "16700": "Charleston–North Charleston, SC",
    "34980": "Nashville, TN",
    "35620": "New York, NY",
    "41860": "San Francisco, CA",
    "19100": "Dallas, TX",
    "28140": "Kansas City, MO–KS",
    "29820": "Las Vegas, NV"
}

# Combine all successfully retrieved years
df_census_historical = pd.concat(
    census_historical_data.values(),
    ignore_index=True
)

# Filter to our selected metropolitan areas
df_census_selected = df_census_historical[
    df_census_historical["cbsa"].isin(
        selected_msas.keys()
    )
].copy()

# Add consistent display names
df_census_selected["selected_msa"] = (
    df_census_selected["cbsa"].map(selected_msas)
)

# Create a coverage matrix
coverage_matrix = pd.crosstab(
    df_census_selected["selected_msa"],
    df_census_selected["year"]
)

print("Selected MSA Historical Coverage:")
display(coverage_matrix)

# Summarize available years for each MSA
coverage_summary = (
    df_census_selected
    .groupby(["cbsa", "selected_msa"])
    .agg(
        available_years=("year", "nunique"),
        first_year=("year", "min"),
        last_year=("year", "max"),
        missing_income=("median_household_income",
                        lambda x: (x <= 0).sum()),
        missing_population=("population",
                            lambda x: (x <= 0).sum())
    )
    .reset_index()
)

print("\nHistorical Coverage Summary:")
display(coverage_summary)

print("\nTotal Selected Observations:",
      len(df_census_selected))

Selected MSA Historical Coverage:


year,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2021,2022,2023,2024
selected_msa,,,,,,,,,,,,,,,,,,,
"Charleston–North Charleston, SC",1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
"Dallas, TX",1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
"Kansas City, MO–KS",1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
"Las Vegas, NV",1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
"Nashville, TN",1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
"New York, NY",1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
"San Francisco, CA",1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1



Historical Coverage Summary:


,cbsa,selected_msa,available_years,first_year,last_year,missing_income,missing_population
0,16700,"Charleston–North Charleston, SC",19,2005,2024,0,0
1,19100,"Dallas, TX",19,2005,2024,0,0
2,28140,"Kansas City, MO–KS",19,2005,2024,0,0
3,29820,"Las Vegas, NV",19,2005,2024,0,0
4,34980,"Nashville, TN",19,2005,2024,0,0
5,35620,"New York, NY",19,2005,2024,0,0
6,41860,"San Francisco, CA",19,2005,2024,0,0



Total Selected Observations: 133


In [8]:
# Step 7: Examine historical income and population changes

# Sort by CBSA and year
df_census_trends = df_census_selected.sort_values(
    ["cbsa", "year"]
).copy()

# Calculate changes only for consecutive calendar years
df_census_trends["previous_year"] = (
    df_census_trends.groupby("cbsa")["year"].shift(1)
)

df_census_trends["year_gap"] = (
    df_census_trends["year"] -
    df_census_trends["previous_year"]
)

# Annual percentage change in population
df_census_trends["population_change_pct"] = (
    df_census_trends.groupby("cbsa")["population"]
    .pct_change(fill_method=None) * 100
)

# Annual percentage change in household income
df_census_trends["income_change_pct"] = (
    df_census_trends.groupby("cbsa")[
        "median_household_income"
    ].pct_change(fill_method=None) * 100
)

# Avoid treating 2019–2021 as a one-year change
df_census_trends.loc[
    df_census_trends["year_gap"] != 1,
    ["population_change_pct", "income_change_pct"]
] = float("nan")

# Review largest absolute changes
print("Largest Population Changes:")
display(
    df_census_trends.loc[
        df_census_trends["population_change_pct"].notna(),
        ["selected_msa", "year", "population_change_pct"]
    ]
    .assign(
        absolute_change=lambda x:
        x["population_change_pct"].abs()
    )
    .sort_values("absolute_change", ascending=False)
    .head(10)
)

print("\nLargest Household Income Changes:")
display(
    df_census_trends.loc[
        df_census_trends["income_change_pct"].notna(),
        ["selected_msa", "year", "income_change_pct"]
    ]
    .assign(
        absolute_change=lambda x:
        x["income_change_pct"].abs()
    )
    .sort_values("absolute_change", ascending=False)
    .head(10)
)

Largest Population Changes:


,selected_msa,year,population_change_pct,absolute_change
3215,"Nashville, TN",2013,6.792867,6.792867
597,"Nashville, TN",2006,5.125088,5.125088
552,"Las Vegas, NV",2006,5.104384,5.104384
446,"Dallas, TX",2006,4.866142,4.866142
424,"Charleston–North Charleston, SC",2006,4.846366,4.846366
965,"Nashville, TN",2007,4.566425,4.566425
791,"Charleston–North Charleston, SC",2007,4.463359,4.463359
3219,"New York, NY",2013,4.120444,4.120444
5552,"New York, NY",2019,-3.820395,3.820395
920,"Las Vegas, NV",2007,3.307607,3.307607



Largest Household Income Changes:


,selected_msa,year,income_change_pct,absolute_change
6283,"Las Vegas, NV",2022,11.181431,11.181431
6399,"San Francisco, CA",2022,10.470238,10.470238
4119,"Nashville, TN",2015,10.153875,10.153875
6332,"Nashville, TN",2022,10.050189,10.050189
5364,"Charleston–North Charleston, SC",2019,9.679075,9.679075
4258,"San Francisco, CA",2016,9.217334,9.217334
6171,"Dallas, TX",2022,9.013491,9.013491
5499,"Las Vegas, NV",2019,8.814563,8.814563
6148,"Charleston–North Charleston, SC",2022,8.536971,8.536971
6336,"New York, NY",2022,8.474215,8.474215


In [9]:
# Step 8: Investigate potential population discontinuities

# Select observations for further review
investigation_cases = {
    "34980": [2012, 2013, 2014],  # Nashville
    "35620": [2012, 2013, 2014, 2018, 2019],
    "16700": [2005, 2006, 2007]   # Charleston
}

for cbsa, years in investigation_cases.items():

    msa_name = selected_msas[cbsa]

    print(f"\n{msa_name}")
    print("-" * 60)

    df_review = df_census_trends[
        (df_census_trends["cbsa"] == cbsa) &
        (df_census_trends["year"].isin(years))
    ][[
        "year",
        "metro",
        "population",
        "population_change_pct",
        "median_household_income",
        "income_change_pct"
    ]].sort_values("year")

    display(df_review)


Nashville, TN
------------------------------------------------------------


,year,metro,population,population_change_pct,median_household_income,income_change_pct
2830,2012,"Nashville-Davidson--Murfreesboro--Franklin, TN...",1645638,1.557329,51500,3.016483
3215,2013,"Nashville-Davidson--Murfreesboro--Franklin, TN...",1757424,6.792867,51996,0.963107
3603,2014,"Nashville-Davidson--Murfreesboro--Franklin, TN...",1792468,1.994055,52640,1.238557



New York, NY
------------------------------------------------------------


,year,metro,population,population_change_pct,median_household_income,income_change_pct
2833,2012,"New York-Northern New Jersey-Long Island, NY-N...",19160024,0.757913,63982,2.663586
3219,2013,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",19949502,4.120444,65786,2.819543
3607,2014,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",20092883,0.718720,67066,1.945703
5162,2018,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",19979477,-1.680041,78478,4.126420
5552,2019,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",19216182,-3.820395,83160,5.966003



Charleston–North Charleston, SC
------------------------------------------------------------


,year,metro,population,population_change_pct,median_household_income,income_change_pct
222,2005,"Charleston-North Charleston, SC Metro Area",575297,NaN,45269,NaN
424,2006,"Charleston-North Charleston, SC Metro Area",603178,4.846366,46421,2.544788
791,2007,"Charleston-North Charleston, SC Metro Area",630100,4.463359,49828,7.339351


In [10]:
# Step 9A: Compare population changes across all seven MSAs

# Select years surrounding the suspected boundary changes
years_to_review = [2012, 2013, 2014, 2018, 2019]

df_boundary_review = df_census_trends[
    df_census_trends["year"].isin(years_to_review)
].copy()

# Create comparison table
population_comparison = df_boundary_review.pivot(
    index="selected_msa",
    columns="year",
    values="population_change_pct"
)

print("Population Changes Around Potential Boundary Revisions:")
display(population_comparison.round(2))

# Identify unusually large changes
print("\nPopulation Changes Greater Than 3% (Absolute):")

large_changes = df_boundary_review[
    df_boundary_review["population_change_pct"].abs() > 3
][[
    "selected_msa",
    "year",
    "population",
    "population_change_pct"
]].sort_values(
    ["year", "selected_msa"]
)

display(large_changes)

Population Changes Around Potential Boundary Revisions:


year,2012,2013,2014,2018,2019
selected_msa,,,,,
"Charleston–North Charleston, SC",2.25,2.12,2.17,1.52,1.84
"Dallas, TX",1.85,2.48,2.08,1.89,0.43
"Kansas City, MO–KS",0.71,-0.59,0.89,0.73,0.59
"Las Vegas, NV",1.56,1.35,2.06,1.25,1.57
"Nashville, TN",1.56,6.79,1.99,1.46,0.09
"New York, NY",0.76,4.12,0.72,-1.68,-3.82
"San Francisco, CA",1.47,1.36,1.72,0.04,0.05



Population Changes Greater Than 3% (Absolute):


,selected_msa,year,population,population_change_pct
3215,"Nashville, TN",2013,1757424,6.792867
3219,"New York, NY",2013,19949502,4.120444
5552,"New York, NY",2019,19216182,-3.820395


In [11]:
# Step 10A: Identify missing ACS years for each selected MSA

expected_years = set(range(2005, 2026))

missing_years_summary = []

for cbsa, msa_name in selected_msas.items():
    available_years = set(
        df_census_selected.loc[
            df_census_selected["cbsa"] == cbsa, "year"
        ]
    )

    missing_years = sorted(expected_years - available_years)

    missing_years_summary.append({
        "cbsa": cbsa,
        "msa": msa_name,
        "available_years": len(available_years),
        "missing_years": missing_years
    })

df_missing_years = pd.DataFrame(missing_years_summary)

display(df_missing_years)

,cbsa,msa,available_years,missing_years
0,16700,"Charleston–North Charleston, SC",19,"[2020, 2025]"
1,34980,"Nashville, TN",19,"[2020, 2025]"
2,35620,"New York, NY",19,"[2020, 2025]"
3,41860,"San Francisco, CA",19,"[2020, 2025]"
4,19100,"Dallas, TX",19,"[2020, 2025]"
5,28140,"Kansas City, MO–KS",19,"[2020, 2025]"
6,29820,"Las Vegas, NV",19,"[2020, 2025]"
